In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
import random
import shutil
import yaml

ROOT = Path.cwd()
LABELS_DIR = ROOT / "labels"
CLASSES_FILE = ROOT / "classes.txt"
ORIG_IMAGES_DIR = Path(r"E:\\SosedTemporary\\WorkingSpace\\Narabotki\\ORIG_valid_images")
WORK_DIR = ROOT / "yolo_dataset_1_0_3"
DATASET_DIR = WORK_DIR / "dataset"
RUNS_DIR = WORK_DIR / "runs"
RUN_NAME = "train"
METRICS_TXT_PATH = WORK_DIR / "metrics_log.txt"
SELECTED_SOURCE_CLASS_IDS = [1, 0, 3]
TRAIN_RATIO = 0.8
SEED = 42
EPOCHS = 50
IMGSZ = 640
BATCH = 16
WORKERS = 0
MODEL_WEIGHTS = "yolo11n.pt"
DEVICE = "cuda"
OVERWRITE_WORK_DIR = True


In [2]:
all_class_names = [line.strip() for line in CLASSES_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
selected_names = [all_class_names[i] for i in SELECTED_SOURCE_CLASS_IDS]
id_map = {src_id: dst_id for dst_id, src_id in enumerate(SELECTED_SOURCE_CLASS_IDS)}

if WORK_DIR.exists() and OVERWRITE_WORK_DIR:
    shutil.rmtree(WORK_DIR)

(DATASET_DIR / "images" / "train").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "images" / "val").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "labels" / "train").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "labels" / "val").mkdir(parents=True, exist_ok=True)

prepared_items = []
for label_path in sorted(LABELS_DIR.glob("*.txt")):
    stem_parts = label_path.stem.split("-")
    if len(stem_parts) != 2:
        continue
    image_id = stem_parts[1]
    image_path = ORIG_IMAGES_DIR / f"{image_id}.png"
    if not image_path.exists():
        continue

    kept = []
    for raw_line in label_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        cls = int(parts[0])
        if cls in id_map:
            parts[0] = str(id_map[cls])
            kept.append(" ".join(parts))

    if kept:
        prepared_items.append((image_id, image_path, kept))

random.seed(SEED)
random.shuffle(prepared_items)
split_idx = int(len(prepared_items) * TRAIN_RATIO)
train_items = prepared_items[:split_idx]
val_items = prepared_items[split_idx:]

for split_name, items in (("train", train_items), ("val", val_items)):
    for image_id, src_image, lines in items:
        dst_image = DATASET_DIR / "images" / split_name / f"{image_id}.png"
        dst_label = DATASET_DIR / "labels" / split_name / f"{image_id}.txt"
        shutil.copy2(src_image, dst_image)
        dst_label.write_text("\n".join(lines) + "\n", encoding="utf-8")

data_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": len(selected_names),
    "names": selected_names,
}
(WORK_DIR / "data.yaml").write_text(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")

{
    "total_labeled_images": len(prepared_items),
    "train_images": len(train_items),
    "val_images": len(val_items),
    "selected_source_class_ids": SELECTED_SOURCE_CLASS_IDS,
    "selected_class_names": selected_names,
    "data_yaml": str((WORK_DIR / "data.yaml").resolve()),
}


{'total_labeled_images': 2500,
 'train_images': 2000,
 'val_images': 500,
 'selected_source_class_ids': [1, 0, 3],
 'selected_class_names': ['лицо', 'голова', 'персонаж'],
 'data_yaml': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\data.yaml'}

In [3]:
import os
import csv
import json
from pathlib import Path

from ultralytics import YOLO
from ultralytics.utils import LOGGER


def _to_float(value):
    try:
        return float(value)
    except Exception:
        return None


def _fmt(value):
    if value is None:
        return "nan"
    return f"{value:.6f}"


def _read_last_row(csv_path: Path):
    if not csv_path.exists():
        return {}
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return {}
    return rows[-1]


def _get_row_value(row, candidates):
    for key in candidates:
        if key in row and row[key] not in ("", None):
            return _to_float(row[key])
    return None


def _extract_overall(row):
    return {
        "train_box_loss": _get_row_value(row, ["train/box_loss"]),
        "train_cls_loss": _get_row_value(row, ["train/cls_loss"]),
        "train_dfl_loss": _get_row_value(row, ["train/dfl_loss"]),
        "val_box_loss": _get_row_value(row, ["val/box_loss"]),
        "val_cls_loss": _get_row_value(row, ["val/cls_loss"]),
        "val_dfl_loss": _get_row_value(row, ["val/dfl_loss"]),
        "precision": _get_row_value(row, ["metrics/precision(B)", "metrics/precision"]),
        "recall": _get_row_value(row, ["metrics/recall(B)", "metrics/recall"]),
        "map50": _get_row_value(row, ["metrics/mAP50(B)", "metrics/mAP50"]),
        "map50_95": _get_row_value(row, ["metrics/mAP50-95(B)", "metrics/mAP50-95"]),
    }


def _extract_box_metrics(trainer):
    validator = getattr(trainer, "validator", None)
    if validator is None:
        return None
    metrics = getattr(validator, "metrics", None)
    if metrics is None:
        return None
    return getattr(metrics, "box", None)


def _emit(text):
    print(text, flush=True)
    LOGGER.info(text)


def _on_train_epoch_end(trainer):
    epoch = int(getattr(trainer, "epoch", -1)) + 1
    _emit(f"train_epoch_end={epoch}/{EPOCHS}")


def _on_fit_epoch_end(trainer):
    epoch = int(getattr(trainer, "epoch", -1)) + 1
    csv_path = Path(trainer.save_dir) / "results.csv"
    row = _read_last_row(csv_path)
    overall = _extract_overall(row)

    box = _extract_box_metrics(trainer)
    if box is not None and hasattr(box, "mean_results"):
        try:
            mean_p, mean_r, mean_map50, mean_map50_95 = box.mean_results()
            if overall["precision"] is None:
                overall["precision"] = _to_float(mean_p)
            if overall["recall"] is None:
                overall["recall"] = _to_float(mean_r)
            if overall["map50"] is None:
                overall["map50"] = _to_float(mean_map50)
            if overall["map50_95"] is None:
                overall["map50_95"] = _to_float(mean_map50_95)
        except Exception:
            pass

    per_class = []
    if box is not None and hasattr(box, "class_result"):
        for class_id, class_name in enumerate(selected_names):
            try:
                class_p, class_r, class_map50, class_map50_95 = box.class_result(class_id)
            except Exception:
                class_p, class_r, class_map50, class_map50_95 = None, None, None, None
            per_class.append(
                {
                    "class_id": class_id,
                    "class_name": class_name,
                    "precision": _to_float(class_p),
                    "recall": _to_float(class_r),
                    "map50": _to_float(class_map50),
                    "map50_95": _to_float(class_map50_95),
                }
            )

    record = {
        "epoch": epoch,
        **overall,
        "per_class": per_class,
    }

    with METRICS_TXT_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    _emit(
        f"epoch={epoch}/{EPOCHS} "
        f"P={_fmt(overall['precision'])} "
        f"R={_fmt(overall['recall'])} "
        f"mAP50={_fmt(overall['map50'])} "
        f"mAP50-95={_fmt(overall['map50_95'])} "
        f"train_box={_fmt(overall['train_box_loss'])} "
        f"train_cls={_fmt(overall['train_cls_loss'])} "
        f"train_dfl={_fmt(overall['train_dfl_loss'])} "
        f"val_box={_fmt(overall['val_box_loss'])} "
        f"val_cls={_fmt(overall['val_cls_loss'])} "
        f"val_dfl={_fmt(overall['val_dfl_loss'])}"
    )

    for cls in per_class:
        _emit(
            f"  class_id={cls['class_id']} "
            f"class_name={cls['class_name']} "
            f"P={_fmt(cls['precision'])} "
            f"R={_fmt(cls['recall'])} "
            f"mAP50={_fmt(cls['map50'])} "
            f"mAP50-95={_fmt(cls['map50_95'])}"
        )


METRICS_TXT_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_TXT_PATH.write_text("", encoding="utf-8")

model = YOLO(MODEL_WEIGHTS)
model.add_callback("on_train_epoch_end", _on_train_epoch_end)
model.add_callback("on_fit_epoch_end", _on_fit_epoch_end)
_emit("custom_progress_logging_enabled")

result = model.train(
    data=str((WORK_DIR / "data.yaml").resolve()),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    project=str(RUNS_DIR.resolve()),
    name=RUN_NAME,
    device=DEVICE,
    verbose=True,
)

save_dir = Path(result.save_dir)
artifacts = {
    "save_dir": str(save_dir.resolve()),
    "best_weights": str((save_dir / "weights" / "best.pt").resolve()),
    "last_weights": str((save_dir / "weights" / "last.pt").resolve()),
    "results_csv": str((save_dir / "results.csv").resolve()),
    "metrics_txt": str(METRICS_TXT_PATH.resolve()),
}
artifacts


custom_progress_logging_enabled
custom_progress_logging_enabled
New https://pypi.org/project/ultralytics/8.4.50 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.245  Python-3.10.19 torch-2.11.0.dev20251231+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\SosedTemporary\WorkingSpace\Narabotki\26-05-14_1\yolo11n_2500\yolo_dataset_1_0_3\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, ma

{'save_dir': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\runs\\train',
 'best_weights': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\runs\\train\\weights\\best.pt',
 'last_weights': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\runs\\train\\weights\\last.pt',
 'results_csv': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\runs\\train\\results.csv',
 'metrics_txt': 'E:\\SosedTemporary\\WorkingSpace\\Narabotki\\26-05-14_1\\yolo11n_2500\\yolo_dataset_1_0_3\\metrics_log.txt'}